#📘 Sistemas Baseados em Conhecimento — Aula 5: Decisão Gradual e Incerteza

##🏭 Case: Monitoramento de Impressão 3D Industrial

Nas Aulas 3 e 4, construímos um motor que decide entre **GO**, **NOGO** e **INVESTIGAR**.

Hoje atacamos um limite real desse sistema:

> **Decisões industriais raramente são binárias.**
> O mundo real vive entre os extremos — e um sistema especialista maduro precisa refletir isso.

Ao final desta aula, você vai:
- Entender por que o limiar rígido (ex: `confianca > 0.80`) é um problema
- Implementar um **score de risco acumulado** que combina múltiplos sinais fracos
- Criar **5 ações graduais** no lugar de 2 extremos
- Testar como o sistema responde de forma proporcional ao risco real

> **O motor da Aula 4 não muda. Toda a evolução está na Base de Conhecimento.**

> **Ontologia e fatos seguem rigorosamente a Aula 4 (predicados simbólicos).**

## 1. Setup do notebook

**O que vamos fazer:** importar as ferramentas básicas — as mesmas usadas na Aula 4.

**Por que vamos fazer:** a premissa central desta aula é que o motor é um executor neutro — a inteligência está nas regras. Não precisamos de um motor novo, precisamos de uma Base de Conhecimento mais madura.

✅ Execute esta célula uma única vez.

In [1]:
from dataclasses import dataclass
from typing import List, Dict, Set, Optional, Tuple
import json

## 2. Base de Fatos (Aula 4) — a "entrada" do motor

**O que vamos fazer:** carregar os mesmos casos F1…F6 da Aula 4.

**Por que vamos fazer:** o motor começa com fatos observados (sensores/grounding).
Mantemos **rigorosamente** a ontologia da Aula 4 — mesmos predicados, mesmos valores — porque toda a BK depende dessa consistência.

📌 Pense nisso como o "estado inicial" do mundo para cada caso. Nada muda aqui.

✅ Execute esta célula para ter os fatos disponíveis.

In [2]:
FATOS: Dict[str, List[str]] = {
    "F1": ["evento_visao(stringing)",     "confianca(baixa)", "vibracao(normal)",  "criticidade(media)", "prazo(longo)"],
    "F2": ["evento_visao(layer_shift)",   "confianca(alta)",  "vibracao(warning)", "criticidade(alta)",  "prazo(curto)"],
    "F3": ["evento_visao(normal)",        "confianca(baixa)", "vibracao(erro)",    "criticidade(media)", "prazo(longo)"],  # conflito
    "F4": ["evento_visao(under_extrusion)","confianca(baixa)","vibracao(normal)",  "criticidade(baixa)", "prazo(longo)"],
    "F5": ["evento_visao(stringing)",     "confianca(baixa)", "vibracao(warning)", "criticidade(alta)",  "prazo(longo)"],
    "F6": ["evento_visao(layer_shift)",   "confianca(alta)",  "vibracao(normal)",  "criticidade(alta)",  "prazo(longo)"],
}

print("Exemplo F3:", FATOS["F3"])
print("Exemplo F5:", FATOS["F5"])

Exemplo F3: ['evento_visao(normal)', 'confianca(baixa)', 'vibracao(erro)', 'criticidade(media)', 'prazo(longo)']
Exemplo F5: ['evento_visao(stringing)', 'confianca(baixa)', 'vibracao(warning)', 'criticidade(alta)', 'prazo(longo)']


## 3. Motor da Aula 4 — sem alterações

**O que vamos fazer:** copiar exatamente o motor da Aula 4 (WorkingMemory + Rule + InferenceEngine).

**Por que vamos fazer:** demonstrar concretamente que **evoluir o sistema não exige alterar o motor**. Um bom motor de inferência é genérico — ele executa qualquer base de regras que você der a ele.

📌 Você **não deve editar nada** nesta célula. O motor é o mesmo de antes.

✅ Execute uma vez. É a base de tudo.

In [3]:
# ============================================================
# MOTOR DE INFERÊNCIA — AULA 4 (não alterar)
# ============================================================

# dataclass : evita código repetitivo
# frozen : torna os objetos imutáveis
@dataclass(frozen=True)

# A Rule é um especialista encapsulado.
# Ela sabe quando deve agir (condições),
# o que deve produzir (ação),
# qual sua força na decisão (prioridade),
# e por que existe (justificativa).
class Rule:
    nome: str          # nome único da regra — identificação no log e na explicação (por_que)
    condicoes: List[str]  # fatos que precisam estar na WM para a regra ser ativada (MATCH) "SE"
    acao: str          # fato gerado ao disparar (FIRE) "ENTÃO"
    prioridade: int    # resolve conflitos: maior prioridade vence (SELECT)
    justificativa: str # rastreabilidade e auditoria


# A WorkingMemory é a "lousa" do sistema.
# Ela começa com os fatos observados.
# E vai acumulando novos fatos conforme as regras disparam.
class WorkingMemory:

    # método que recebe os fatos observados inicialmente (vindos do grounding / sensores)
    def __init__(self, fatos_iniciais: List[str]):
        self.fatos: Set[str] = set(fatos_iniciais)

    # adiciona um novo fato à memória; retorna True se adicionado, False se já existia
    def add(self, fato: str) -> bool:
        if fato in self.fatos:
            return False
        self.fatos.add(fato)
        return True

    # verifica se TODAS as condições de uma regra estão na WM — etapa MATCH
    def contains_all(self, conds: List[str]) -> bool:
        return all(c in self.fatos for c in conds)


# A InferenceEngine é o "orquestrador" do raciocínio.
# Ela executa o ciclo MATCH → SELECT → FIRE:
# 1) MATCH: encontra quais regras podem disparar (agenda)
# 2) SELECT: escolhe a regra vencedora (maior prioridade)
# 3) FIRE:   dispara a regra e adiciona um novo fato na WM
# Repete até não haver mais regras ativáveis (convergência).
# Registra o caminho (support) para suportar explicação com por_que().
class InferenceEngine:

    # recebe as regras, define a estratégia de conflito (prioridade) e prepara rastreabilidade
    def __init__(self, regras: List[Rule]):
        # ordena regras por prioridade (maior primeiro) — estratégia de resolução de conflito
        self.regras = sorted(regras, key=lambda r: (-r.prioridade, r.nome))
        self.last_fired: List[Tuple[str, str]] = []  # log de regras disparadas (auditoria)
        self.support: Dict[str, Dict] = {}           # fato → {rule, needs} para por_que()

    def executar(self, wm: WorkingMemory, max_ciclos: int = 30, verbose: bool = True) -> Dict:
        self.last_fired = []
        self.support = {}
        ciclos = 0

        while ciclos < max_ciclos:
            ciclos += 1
            agenda = [r for r in self.regras if wm.contains_all(r.condicoes) and r.acao not in wm.fatos]

            if not agenda:
                if verbose:
                    print(f"\nCiclo {ciclos}: agenda vazia → convergiu.")
                break

            # SELECT: maior prioridade (lista já ordenada)
            escolhida = agenda[0]

            if verbose:
                print(f"\nCiclo {ciclos}")
                print("Agenda:", [a.nome for a in agenda[:6]], "..." if len(agenda) > 6 else "")
                print("SELECT →", escolhida.nome, f"(P={escolhida.prioridade})")
                print("FIRE   →", escolhida.acao)

            # FIRE
            added = wm.add(escolhida.acao)
            if added:
                self.last_fired.append((escolhida.nome, escolhida.acao))
                self.support[escolhida.acao] = {"rule": escolhida, "needs": list(escolhida.condicoes)}

        return {"ciclos": ciclos, "wm": wm, "fired": self.last_fired}

    # reconstrói a cadeia causal: qual regra gerou qual fato, e quais fatos sustentaram aquela regra
    def por_que(self, objetivo: str) -> Dict:
        if objetivo not in self.support:
            return {"objetivo": objetivo, "tipo": "observado_ou_ausente",
                    "detalhe": "Fato inicial (observado) ou não derivado pelo motor."}
        node = self.support[objetivo]
        rule: Rule = node["rule"]
        children = [
            self.por_que(need) if need in self.support
            else {"objetivo": need, "tipo": "observado", "detalhe": "Fato inicial na WM."}
            for need in node["needs"]
        ]
        return {
            "objetivo": objetivo,
            "tipo": "derivado",
            "regra": rule.nome,
            "prioridade": rule.prioridade,
            "justificativa": rule.justificativa,
            "suportado_por": children,
        }


print("✅ Motor carregado (Aula 4, sem alterações).")

✅ Motor carregado (Aula 4, sem alterações).


## 4. Base de Conhecimento (Aula 4) — ponto de partida

**O que vamos fazer:** carregar a BASE_REGRAS e os helpers exatamente como estavam na Aula 4.

**Por que vamos fazer:** antes de mostrar o problema, precisamos ter o sistema anterior funcionando — para comparar depois e entender o que vai mudar.

📌 Importante:
- Regras geram fatos intermediários (camada 1) e decisões (camada 2)
- Conflitos são resolvidos por **prioridade** (critério auditável)

✅ Execute para carregar as regras.

In [4]:
# Base de Conhecimento da Aula 4 — mantida sem alterações
BASE_REGRAS: List[Rule] = [
    # --- Camada 1: interpretação dos sensores ---
    Rule(
        nome="R01_VIBRACAO_ERRO_RISCO_ALTO",
        condicoes=["vibracao(erro)"],
        acao="risco_mecanico(alto)",
        prioridade=95,
        justificativa="Vibração em erro indica anomalia mecânica grave."
    ),
    Rule(
        nome="R02_VIBRACAO_WARNING_RISCO_MEDIO",
        condicoes=["vibracao(warning)"],
        acao="risco_mecanico(medio)",
        prioridade=65,
        justificativa="Warning de vibração indica risco mecânico moderado."
    ),
    Rule(
        nome="R03_LAYER_SHIFT_CONFIRMADO",
        condicoes=["evento_visao(layer_shift)", "confianca(alta)"],
        acao="defeito_confirmado(layer_shift)",
        prioridade=88,
        justificativa="Layer shift com confiança alta é defeito confirmado."
    ),
    Rule(
        nome="R04_STRINGING_SUSPEITO",
        condicoes=["evento_visao(stringing)", "confianca(baixa)"],
        acao="defeito_suspeito(stringing)",
        prioridade=55,
        justificativa="Stringing com confiança baixa é evidência fraca → suspeito."
    ),
    Rule(
        nome="R05_UNDER_EXTRUSION_SUSPEITO",
        condicoes=["evento_visao(under_extrusion)", "confianca(baixa)"],
        acao="defeito_suspeito(under_extrusion)",
        prioridade=55,
        justificativa="Under extrusion com baixa confiança é evidência fraca → tratar como suspeito."
    ),
    # --- Camada 2: decisão ---
    Rule(
        nome="R07_NOGO_LAYER_SHIFT_CRITICO",
        condicoes=["defeito_confirmado(layer_shift)", "criticidade(alta)"],
        acao="decisao(NOGO)",
        prioridade=100,
        justificativa="Defeito confirmado em peça crítica → parar (NOGO)."
    ),
    Rule(
        nome="R08_NOGO_RISCO_MECANICO_ALTO",
        condicoes=["risco_mecanico(alto)"],
        acao="decisao(NOGO)",
        prioridade=97,
        justificativa="Risco mecânico alto → parar por segurança (NOGO)."
    ),
    Rule(
        nome="R09_INVESTIGAR_SUSPEITO_CRITICO",
        condicoes=["defeito_suspeito(stringing)", "criticidade(alta)"],
        acao="decisao(INVESTIGAR)",
        prioridade=68,
        justificativa="Suspeita de defeito em peça crítica → investigar antes de seguir."
    ),
    Rule(
        nome="R10_GO_SENSORES_NORMAIS",
        condicoes=["evento_visao(normal)", "vibracao(normal)"],
        acao="decisao(GO)",
        prioridade=10,
        justificativa="Sem defeitos e vibração normal → seguir (GO)."
    ),
    Rule(
        nome="R11_INVESTIGAR_RISCO_MEDIO_CRITICO",
        condicoes=["risco_mecanico(medio)", "criticidade(alta)"],
        acao="decisao(INVESTIGAR)",
        prioridade=5,
        justificativa="Risco mecânico médio em peça crítica exige investigação antes de seguir."
    ),
    Rule(
        nome="R12_ALERTA_PREVENTIVO_RISCO_MEDIO_CRITICO",
        condicoes=["risco_mecanico(medio)", "criticidade(alta)"],
        acao="alerta_preventivo(ativo)",
        prioridade=5,
        justificativa="Risco médio em peça crítica → ativar alerta preventivo."
    ),
    Rule(
        nome="R13_INVESTIGAR_ALERTA_PREVENTIVO",
        condicoes=["alerta_preventivo(ativo)", "defeito_suspeito(stringing)"],
        acao="decisao(INVESTIGAR)",
        prioridade=10,
        justificativa="Alerta preventivo + suspeita de defeito → investigar."
    ),
]

# Helpers da Aula 4 — mantidos sem alteração
def rodar_caso(case_id: str, regras: List[Rule] = BASE_REGRAS, verbose: bool = True):
    wm = WorkingMemory(FATOS[case_id])
    eng = InferenceEngine(regras)
    resultado = eng.executar(wm, verbose=verbose)
    return eng, resultado

def mostrar_resumo(case_id: str, resultado: Dict):
    print("\n--- RESUMO ---")
    print("Caso:", case_id)
    print("WM final (ordenada):")
    for f in sorted(list(resultado["wm"].fatos)):
        print(" -", f)
    decis = [f for f in resultado["wm"].fatos if f.startswith("decisao(") or f.startswith("acao(")]
    print("\nDecisão:", decis[0] if decis else "(nenhuma)")

print("Regras carregadas:", len(BASE_REGRAS))

Regras carregadas: 12


## 5. O problema do limiar rígido — DEMONSTRAÇÃO

**O que vamos fazer:** mostrar concretamente o que acontece quando dois casos quase idênticos recebem decisões radicalmente diferentes.

**Por que vamos fazer:** antes de propor uma solução, precisamos sentir o problema com os nossos próprios casos F1–F6.

📌 Os dois casos abaixo diferem em apenas **um símbolo de confiança** — mas no sistema binário isso muda tudo.

✅ Execute e observe a diferença.

In [5]:
# Sistema binário da Aula 4 — demonstração do problema

# Usamos uma BK SIMPLIFICADA para isolar o efeito do limiar de confiança - EXEMPLO
BK_BINARIA = [
    Rule(
        nome="R_CONF_ALTA",
        condicoes=["confianca(alta)"],          # tudo ou nada
        acao="sinal_confiavel(sim)",
        prioridade=80,
        justificativa="Confiança marcada como alta → sinal confiável."
    ),
    Rule(
        nome="R_STRINGING_CONFIRMADO",
        condicoes=["evento_visao(stringing)", "sinal_confiavel(sim)"],
        acao="defeito_confirmado(stringing)",
        prioridade=75,
        justificativa="Stringing com confiança alta → defeito confirmado."
    ),
    Rule(
        nome="R_NOGO_STRINGING_CRITICO",
        condicoes=["defeito_confirmado(stringing)", "criticidade(alta)"],
        acao="decisao(NOGO)",
        prioridade=90,
        justificativa="Defeito confirmado em peça crítica → NOGO."
    ),
]

# Caso A: confiança = 0.82 → rotulado como 'alta' (acima do corte)
caso_A = ["evento_visao(stringing)", "confianca(alta)", "criticidade(alta)", "prazo(longo)"]

# Caso B: confiança = 0.79 → rotulado como 'baixa' — apenas 0.03 de diferença!
caso_B = ["evento_visao(stringing)", "confianca(baixa)", "criticidade(alta)", "prazo(longo)"]

print("=" * 55)
print("CASO A — confiança 0.82 (rotulado como 'alta')")
wm_A = WorkingMemory(caso_A)
eng_A = InferenceEngine(BK_BINARIA)
res_A = eng_A.executar(wm_A, verbose=False)
acao_A = next((f for f in res_A["wm"].fatos if f.startswith("decisao(") or f.startswith("acao(")), "(nenhuma)")
print(f"🎯 Resultado: {acao_A}")

print("\n" + "=" * 55)
print("CASO B — confiança 0.79 (rotulado como 'baixa')")
wm_B = WorkingMemory(caso_B)
eng_B = InferenceEngine(BK_BINARIA)
res_B = eng_B.executar(wm_B, verbose=False)
acao_B = next((f for f in res_B["wm"].fatos if f.startswith("decisao(") or f.startswith("acao(")), "(nenhuma)")
print(f"🎯 Resultado: {acao_B}")

print("\n" + "=" * 55)
print("⚠  Diferença de 0.03 na confiança → decisões completamente diferentes.")
print("   Isso é o problema do limiar rígido.")

CASO A — confiança 0.82 (rotulado como 'alta')
🎯 Resultado: decisao(NOGO)

CASO B — confiança 0.79 (rotulado como 'baixa')
🎯 Resultado: (nenhuma)

⚠  Diferença de 0.03 na confiança → decisões completamente diferentes.
   Isso é o problema do limiar rígido.


### 🔎 O que aconteceu?

O Caso A recebeu **NOGO** e o Caso B ficou **sem decisão** — por uma diferença de 0.03 no valor de confiança.

No mundo real, `0.82` e `0.79` são praticamente a mesma coisa. Mas o sistema binário força uma escolha.

**Três perguntas que esse resultado levanta:**
1. Quem definiu que 0.80 é o corte? Com base em quê?
2. O que acontece com todos os casos que ficam "sem decisão"?
3. Uma decisão NOGO por stringing leve é proporcional ao risco?

---

Além do limiar rígido, o sistema da Aula 4 tem mais dois problemas:

**📊 O problema da proporcionalidade:**
Um stringing leve em peça não-crítica recebe o mesmo NOGO de um layer_shift em peça aeroespacial.
A reação não é proporcional ao risco.

**🔀 O problema da combinação:**
Vários sinais fracos simultâneos podem compor um risco real maior do que qualquer sinal isolado — o sistema binário não captura essa acumulação.

## 6. Solução — Score de Risco + Ações Graduais

**O que vamos fazer:** implementar uma função de score de risco e uma Base de Conhecimento que produz **5 ações graduais**.

**Por que vamos fazer:** em vez de colapsar a confiança em binário, vamos preservar o valor numérico e combinar múltiplos sinais em um score ponderado. A ação final depende da **faixa do score** — não de um único limiar.

📌 A arquitetura:
- **Camada 0:** função Python calcula `score_risco` e injeta na WM como predicado
- **Camada 1:** regras interpretam os sinais (as mesmas da Aula 4)
- **Camada 2:** regras de decisão produzem ações graduais: GO / MONITORAR / AJUSTAR / INVESTIGAR / NOGO

✅ Execute para carregar a infraestrutura.

In [6]:
# ============================================================
# FUNÇÃO DE SCORE DE RISCO
# Alinhada à tabela da aula (slide Score de Risco)
# ============================================================

PESOS_SCORE = {
    # Sinais de visão
    "evento_visao(stringing)":       20,
    "evento_visao(layer_shift)":     45,
    "evento_visao(under_extrusion)": 15,
    # Bônus de confiança (aplicado se confianca_num >= 0.80)
    "confianca_score_acima_0.80":    15,
    # Vibração
    "vibracao(warning)":             20,
    "vibracao(erro)":                40,
    # Contexto operacional
    "criticidade(alta)":             20,
    "criticidade(media)":             8,
    "prazo(curto)":                  10,
}

# Faixas alinhadas ao slide 7 da aula:
# 0–29 → GO | 30–49 → MONITORAR | 50–69 → AJUSTAR | 70–84 → INVESTIGAR | 85+ → NOGO
def calcular_score(fatos: List[str], confianca_num: float = 0.0) -> Tuple[int, str]:
    # Recebe lista de fatos simbólicos e valor numérico de confiança.
    # Retorna (score_int, predicado_para_wm).
    score = sum(PESOS_SCORE.get(f, 0) for f in fatos)
    if confianca_num >= 0.80:
        score += PESOS_SCORE["confianca_score_acima_0.80"]
    score = min(score, 100)  # teto em 100

    if score < 30:
        faixa = "baixo"       # → GO
    elif score < 50:
        faixa = "moderado"    # → MONITORAR
    elif score < 70:
        faixa = "alto"        # → AJUSTAR
    elif score < 85:
        faixa = "critico"     # → INVESTIGAR
    else:
        faixa = "extremo"     # → NOGO

    return score, f"score_risco({faixa})"


# Teste rápido com F5 (stringing + warning + criticidade alta)
fatos_teste = FATOS["F5"]
s, pred = calcular_score(fatos_teste, confianca_num=0.72)
print(f"Score F5: {s}  →  {pred}")
print("\n📌 Esse predicado será injetado na WM antes do motor rodar.")

Score F5: 60  →  score_risco(alto)

📌 Esse predicado será injetado na WM antes do motor rodar.


## 7. Base de Conhecimento Gradual (BK v2)

**O que vamos fazer:** definir a nova BK com 5 camadas de ação, partindo das regras da Aula 4.

**Por que vamos fazer:** a BK v2 usa o `score_risco` como predicado — permitindo que regras se refiram à **faixa de risco** em vez de a sinais individuais. Isso resolve os três problemas identificados (limiar rígido, proporcionalidade, combinação de sinais).

📌 Importante:
- Regras R01–R11 da Aula 4 são **mantidas com suas prioridades originais**
- Novas regras de MONITORAR e AJUSTAR têm prioridade menor — só ativam quando NOGO/INVESTIGAR não se aplicam
- O motor **NÃO muda** — apenas adicionamos regras à Base de Conhecimento (mesmo princípio da Aula 4)

✅ Execute para carregar a BK v2.

In [7]:
# ============================================================
# BASE DE CONHECIMENTO v2 — Ações Graduais
# R01–R13: herdadas da Aula 4 (R01-R08 camada 1, R09-R13 camada 2)
# R14–R21: novas regras graduais (score, AJUSTAR, MONITORAR)
# ============================================================

BK_GRADUAL: List[Rule] = [

    # --- CAMADA 1: Interpretação dos sensores (Aula 4) ---
    Rule(
        nome="R01_VIBRACAO_ERRO_RISCO_ALTO",
        condicoes=["vibracao(erro)"],
        acao="risco_mecanico(alto)",
        prioridade=95,
        justificativa="Vibração em erro indica anomalia mecânica grave."
    ),
    Rule(
        nome="R02_VIBRACAO_WARNING_RISCO_MEDIO",
        condicoes=["vibracao(warning)"],
        acao="risco_mecanico(medio)",
        prioridade=65,
        justificativa="Warning de vibração indica risco mecânico moderado."
    ),
    Rule(
        nome="R03_LAYER_SHIFT_CONFIRMADO",
        condicoes=["evento_visao(layer_shift)", "confianca(alta)"],
        acao="defeito_confirmado(layer_shift)",
        prioridade=88,
        justificativa="Layer shift com confiança alta é defeito confirmado."
    ),
    Rule(
        nome="R04_STRINGING_SUSPEITO",
        condicoes=["evento_visao(stringing)", "confianca(baixa)"],
        acao="defeito_suspeito(stringing)",
        prioridade=55,
        justificativa="Stringing com confiança baixa é evidência fraca → suspeito."
    ),
    Rule(
        nome="R05_UNDER_EXTRUSION_SUSPEITO",
        condicoes=["evento_visao(under_extrusion)", "confianca(baixa)"],
        acao="defeito_suspeito(under_extrusion)",
        prioridade=55,
        justificativa="Under extrusion com baixa confiança → tratar como suspeito."
    ),

    # --- CAMADA 2: Decisões NOGO (prioridade máxima — herdadas da Aula 4) ---
    Rule(
        nome="R07_NOGO_LAYER_SHIFT_CRITICO",
        condicoes=["defeito_confirmado(layer_shift)", "criticidade(alta)"],
        acao="acao(NOGO)",
        prioridade=100,
        justificativa="Defeito confirmado em peça crítica → parar (NOGO)."
    ),
    Rule(
        nome="R08_NOGO_RISCO_MECANICO_ALTO",
        condicoes=["risco_mecanico(alto)"],
        acao="acao(NOGO)",
        prioridade=97,
        justificativa="Risco mecânico alto → parar por segurança (NOGO)."
    ),

    # --- CAMADA 2: INVESTIGAR / GO (herdadas da Aula 4) ---
    Rule(
        nome="R09_INVESTIGAR_SUSPEITO_CRITICO",
        condicoes=["defeito_suspeito(stringing)", "criticidade(alta)"],
        acao="acao(INVESTIGAR)",
        prioridade=68,
        justificativa="Suspeita de defeito em peça crítica → investigar antes de seguir."
    ),
    Rule(
        nome="R10_GO_SENSORES_NORMAIS",
        condicoes=["evento_visao(normal)", "vibracao(normal)"],
        acao="acao(GO)",
        prioridade=10,
        justificativa="Sem defeitos e vibração normal → seguir (GO)."
    ),
    Rule(
        nome="R11_INVESTIGAR_RISCO_MEDIO_CRITICO",
        condicoes=["risco_mecanico(medio)", "criticidade(alta)"],
        acao="acao(INVESTIGAR)",
        prioridade=5,
        justificativa="Risco mecânico médio em peça crítica exige investigação antes de seguir."
    ),
    Rule(
        nome="R12_ALERTA_PREVENTIVO_RISCO_MEDIO_CRITICO",
        condicoes=["risco_mecanico(medio)", "criticidade(alta)"],
        acao="alerta_preventivo(ativo)",
        prioridade=5,
        justificativa="Risco médio em peça crítica → ativar alerta preventivo."
    ),
    Rule(
        nome="R13_INVESTIGAR_ALERTA_PREVENTIVO",
        condicoes=["alerta_preventivo(ativo)", "defeito_suspeito(stringing)"],
        acao="acao(INVESTIGAR)",
        prioridade=10,
        justificativa="Alerta preventivo + suspeita de defeito → investigar."
    ),

    # --- CAMADA 2: NOGO por score extremo (nova) ---
    Rule(
        nome="R14_NOGO_SCORE_EXTREMO",
        condicoes=["score_risco(extremo)"],
        acao="acao(NOGO)",
        prioridade=96,
        justificativa="Score de risco extremo (>=85): combinação de sinais indica risco grave -> NOGO."
    ),

    # --- CAMADA 2: INVESTIGAR por score (nova) ---
    Rule(
        nome="R15_INVESTIGAR_LAYER_SHIFT_NAO_CRITICO",
        condicoes=["defeito_confirmado(layer_shift)", "criticidade(media)"],
        acao="acao(INVESTIGAR)",
        prioridade=85,
        justificativa="Layer shift confirmado em peça não-crítica: investigar antes de continuar."
    ),
    Rule(
        nome="R16_INVESTIGAR_SCORE_CRITICO_PRAZO_LONGO",
        condicoes=["score_risco(critico)", "prazo(longo)"],
        acao="acao(INVESTIGAR)",
        prioridade=72,
        justificativa="Score crítico + prazo longo: há tempo para investigar sem paralisar."
    ),

    # --- CAMADA 2: AJUSTAR (nova) ---
    Rule(
        nome="R17_AJUSTAR_SCORE_ALTO_PRAZO_CURTO",
        condicoes=["score_risco(alto)", "prazo(curto)"],
        acao="acao(AJUSTAR)",
        prioridade=62,
        justificativa="Score alto + prazo curto: sem tempo para parar, ajustar parâmetro agora."
    ),
    Rule(
        nome="R18_AJUSTAR_UNDER_EXTRUSION_MODERADO",
        condicoes=["defeito_suspeito(under_extrusion)", "score_risco(moderado)"],
        acao="acao(AJUSTAR)",
        prioridade=58,
        justificativa="Under-extrusion com risco moderado: corrigir temperatura antes de piorar."
    ),

    # --- CAMADA 2: MONITORAR (novo) ---
    Rule(
        nome="R19_MONITORAR_STRINGING_BAIXO_RISCO",
        condicoes=["defeito_suspeito(stringing)", "score_risco(moderado)", "criticidade(baixa)"],
        acao="acao(MONITORAR)",
        prioridade=45,
        justificativa="Stringing suspeito em peça não-crítica com risco moderado: apenas monitorar."
    ),
    Rule(
        nome="R20_MONITORAR_SCORE_BAIXO",
        condicoes=["score_risco(baixo)"],
        acao="acao(MONITORAR)",
        prioridade=30,
        justificativa="Score baixo: algum sinal presente, mas risco não justifica interrupção."
    ),

    # --- CAMADA 2: GO ---
    Rule(
        nome="R21_GO_SENSORES_NORMAIS",
        condicoes=["evento_visao(normal)", "vibracao(normal)"],
        acao="acao(GO)",
        prioridade=10,
        justificativa="Sem defeitos e vibração normal → seguir (GO)."
    ),
]

print(f"✅ BK Gradual carregada: {len(BK_GRADUAL)} regras")
print("\nDistribuição por ação:")
for acao in ["NOGO", "INVESTIGAR", "AJUSTAR", "MONITORAR", "GO"]:
    qtd = sum(1 for r in BK_GRADUAL if acao in r.acao)
    print(f"  {acao:12s}: {qtd} regra(s)")

✅ BK Gradual carregada: 20 regras

Distribuição por ação:
  NOGO        : 3 regra(s)
  INVESTIGAR  : 5 regra(s)
  AJUSTAR     : 2 regra(s)
  MONITORAR   : 2 regra(s)
  GO          : 5 regra(s)


## 8. Função auxiliar: preparar_e_rodar

**O que vamos fazer:** criar uma função que recebe o `case_id` (F1…F6) + valor numérico de confiança, calcula o score, injeta o predicado na WM e roda o motor.

**Por que vamos fazer:** isso simula o que aconteceria na integração com sensores reais — o score é calculado antes do motor rodar, e o predicado resultante é tratado como mais um fato.

✅ Execute para liberar os atalhos.

In [8]:
def preparar_e_rodar(
    case_id: str,
    confianca_num: float = 0.0,
    regras: List[Rule] = BK_GRADUAL,
    verbose: bool = True
):
    # 1) Carrega FATOS[case_id]
    # 2) Calcula score de risco a partir dos fatos e confianca numérica
    # 3) Injeta predicado score_risco na WM
    # 4) Roda o motor
    # 5) Mostra resumo
    fatos_base = FATOS[case_id]
    score, predicado_score = calcular_score(fatos_base, confianca_num)
    fatos_completos = fatos_base + [predicado_score]

    print(f"\n{'='*55}")
    print(f"CASO: {case_id}")
    print(f"Score calculado: {score}  →  {predicado_score}")
    print(f"Fatos na WM: {fatos_completos}")

    wm = WorkingMemory(fatos_completos)
    eng = InferenceEngine(regras)
    resultado = eng.executar(wm, verbose=verbose)

    print("\n--- RESUMO ---")
    acoes = [f for f in resultado["wm"].fatos if f.startswith("acao(")]
    print(f"🎯 Resultado: {acoes[0] if acoes else '(nenhuma)'}")
    return eng, resultado


# Versão para fatos avulsos (casos customizados fora de F1-F6)
def preparar_e_rodar_fatos(
    label: str,
    fatos_base: List[str],
    confianca_num: float = 0.0,
    regras: List[Rule] = BK_GRADUAL,
    verbose: bool = True
):
    score, predicado_score = calcular_score(fatos_base, confianca_num)
    fatos_completos = fatos_base + [predicado_score]

    print(f"\n{'='*55}")
    print(f"CASO: {label}")
    print(f"Score calculado: {score}  →  {predicado_score}")
    print(f"Fatos na WM: {fatos_completos}")

    wm = WorkingMemory(fatos_completos)
    eng = InferenceEngine(regras)
    resultado = eng.executar(wm, verbose=verbose)

    print("\n--- RESUMO ---")
    acoes = [f for f in resultado["wm"].fatos if f.startswith("acao(")]
    print(f"🎯 Resultado: {acoes[0] if acoes else '(nenhuma)'}")
    return eng, resultado

print("✅ Funções preparar_e_rodar() e preparar_e_rodar_fatos() disponíveis.")

✅ Funções preparar_e_rodar() e preparar_e_rodar_fatos() disponíveis.


## 9. Demonstração 1 — Reanalisando o problema do limiar rígido

**O que vamos fazer:** rodar os mesmos dois casos da Seção 5 (confiança 0.82 vs 0.79) agora com a BK Gradual.

**Por que vamos fazer:** mostrar que o score suaviza a diferença — as duas respostas serão próximas, não opostas.

✅ Execute e compare com o resultado da Seção 5.

In [9]:
# Recriando os casos da seção 5, agora com score acumulado
# Caso A: confiança = 0.82 → rotulado como 'alta' (acima do corte)
fatos_A = ["evento_visao(stringing)", "confianca(alta)", "criticidade(alta)", "prazo(longo)"]

# Caso B: confiança = 0.79 → rotulado como 'baixa' — apenas 0.03 de diferença!
fatos_B = ["evento_visao(stringing)", "confianca(baixa)", "criticidade(alta)", "prazo(longo)"]

score_A, pred_A = calcular_score(fatos_A, confianca_num=0.82)
score_B, pred_B = calcular_score(fatos_B, confianca_num=0.79)

print("=" * 55)
print("CASO A — confiança 0.82")
wm_A = WorkingMemory(fatos_A + [pred_A])
eng_A, res_A = WorkingMemory(fatos_A + [pred_A]), None
eng_A = InferenceEngine(BK_GRADUAL)
res_A = eng_A.executar(WorkingMemory(fatos_A + [pred_A]), verbose=False)
acao_A = next((f for f in res_A["wm"].fatos if f.startswith("acao(")), "(nenhuma)")
print(f"  Score={score_A}  {pred_A}  →  {acao_A}")

print("\nCASO B — confiança 0.79")
eng_B = InferenceEngine(BK_GRADUAL)
res_B = eng_B.executar(WorkingMemory(fatos_B + [pred_B]), verbose=False)
acao_B = next((f for f in res_B["wm"].fatos if f.startswith("acao(")), "(nenhuma)")
print(f"  Score={score_B}  {pred_B}  →  {acao_B}")

print("\n" + "=" * 55)
print("📊 Comparação:")
print(f"  Antes (Aula 4): 0.82 → NOGO  |  0.79 → (sem decisão)")
print(f"  Agora (Aula 5): 0.82 → {acao_A}  |  0.79 → {acao_B}")
print("\n✅ O score absorveu a diferença de 0.03 — respostas agora são proporcionais.")

CASO A — confiança 0.82
  Score=55  score_risco(alto)  →  (nenhuma)

CASO B — confiança 0.79
  Score=40  score_risco(moderado)  →  acao(INVESTIGAR)

📊 Comparação:
  Antes (Aula 4): 0.82 → NOGO  |  0.79 → (sem decisão)
  Agora (Aula 5): 0.82 → (nenhuma)  |  0.79 → acao(INVESTIGAR)

✅ O score absorveu a diferença de 0.03 — respostas agora são proporcionais.


## 10. Demonstração 2 — F1…F6 no espectro de decisão

**O que vamos fazer:** rodar todos os casos F1…F6 com a BK Gradual.

**Por que vamos fazer:** visualizar que o sistema agora responde proporcionalmente ao risco real — usando os **mesmos casos que conhecemos da Aula 4**.

📌 Compare com as decisões binárias que obtínhamos antes.

✅ Execute e observe o espectro completo.

In [10]:
# Confiança numérica coerente com o símbolo confianca(alta/baixa) de cada caso
CONFIANCA_NUM = {
    "F1": 0.65,   # confianca(baixa)
    "F2": 0.88,   # confianca(alta)
    "F3": 0.62,   # confianca(baixa)
    "F4": 0.60,   # confianca(baixa)
    "F5": 0.70,   # confianca(baixa)
    "F6": 0.91,   # confianca(alta)
}

print("ESPECTRO DE DECISÃO — F1…F6 COM BK GRADUAL")
print("=" * 65)
print(f"{'Caso':<5} {'Score':>5}  {'Faixa':<22} {'Ação Gradual':<20}")
print("-" * 65)

for fid in ["F1", "F2", "F3", "F4", "F5", "F6"]:
    conf = CONFIANCA_NUM[fid]
    score, pred = calcular_score(FATOS[fid], conf)
    fatos_c = FATOS[fid] + [pred]
    wm = WorkingMemory(fatos_c)
    eng_tmp = InferenceEngine(BK_GRADUAL)
    res_tmp = eng_tmp.executar(wm, verbose=False)
    acao = next((f for f in res_tmp["wm"].fatos if f.startswith("acao(")), "(nenhuma)")
    print(f"  {fid}  {score:>5}  {pred:<22} {acao}")

ESPECTRO DE DECISÃO — F1…F6 COM BK GRADUAL
Caso  Score  Faixa                  Ação Gradual        
-----------------------------------------------------------------
  F1     28  score_risco(baixo)     acao(MONITORAR)
  F2    100  score_risco(extremo)   acao(NOGO)
  F3     48  score_risco(moderado)  acao(NOGO)
  F4     15  score_risco(baixo)     acao(MONITORAR)
  F5     60  score_risco(alto)      acao(INVESTIGAR)
  F6     80  score_risco(critico)   acao(NOGO)


## 11. Demonstração 3 — Caso G1 passo a passo

**O que vamos fazer:** rodar o caso G1 exatamente como apresentado no slide da aula — da leitura dos sensores ao score, e do score à ação.

**Por que vamos fazer:** tornar o mecanismo do score completamente transparente, passo a passo.

📌 G1: `stringing` + `vibracao(warning)` + `criticidade(alta)` + `prazo(longo)` + `confianca_num=0.72`

✅ Execute e acompanhe o cálculo.

In [11]:
# Caso G1 do slide — cálculo passo a passo
fatos_G1 = [
    "evento_visao(stringing)",  # +20
    "vibracao(warning)",         # +20
    "criticidade(alta)",         # +20
    "prazo(longo)",              # +0
]
confianca_G1 = 0.72  # < 0.80 → bônus = 0

print("=== Caso G1 — cálculo passo a passo ===\n")
detalhes = {
    "evento_visao(stringing)": 20,
    "vibracao(warning)": 20,
    "criticidade(alta)": 20,
    "prazo(longo)": 0,
}
total = 0
for fato, peso in detalhes.items():
    print(f"  {fato:<32} +{peso}")
    total += peso
bonus = 15 if confianca_G1 >= 0.80 else 0
bonus_str = "(>= 0.80 +15)" if bonus else "(< 0.80  +0 )"
print(f"  confianca={confianca_G1} {bonus_str}   +{bonus}")
total += bonus
print("  " + chr(9472)*44)
print(f"  SCORE TOTAL = {total}")
print()
sc, pred = calcular_score(fatos_G1, confianca_G1)
faixas_map = {"baixo":"0-29 GO","moderado":"30-49 MONITORAR","alto":"50-69 AJUSTAR",
              "critico":"70-84 INVESTIGAR","extremo":"85+ NOGO"}
chave = pred.split("(")[1][:-1]
print(f"  Faixa: {faixas_map.get(chave, pred)}")

# Rodar motor
print()
eng_G1, res_G1 = preparar_e_rodar_fatos("G1", fatos_G1, confianca_num=confianca_G1, verbose=False)
acao_G1 = next((f for f in res_G1["wm"].fatos if f.startswith("acao(")), "(nenhuma)")
print(f"\n🔧 Decisão final: {acao_G1}")

=== Caso G1 — cálculo passo a passo ===

  evento_visao(stringing)          +20
  vibracao(warning)                +20
  criticidade(alta)                +20
  prazo(longo)                     +0
  confianca=0.72 (< 0.80  +0 )   +0
  ────────────────────────────────────────────
  SCORE TOTAL = 60

  Faixa: 50-69 AJUSTAR


CASO: G1
Score calculado: 60  →  score_risco(alto)
Fatos na WM: ['evento_visao(stringing)', 'vibracao(warning)', 'criticidade(alta)', 'prazo(longo)', 'score_risco(alto)']

--- RESUMO ---
🎯 Resultado: acao(INVESTIGAR)

🔧 Decisão final: acao(INVESTIGAR)


## 12. Demonstração 4 — Sinais fracos que se acumulam

**O que vamos fazer:** mostrar que três sinais fracos juntos podem compor um risco alto que nenhum deles expressaria sozinho.

**Por que vamos fazer:** esse é um dos principais benefícios do score acumulado — capturar **risco composto**.

✅ Execute e observe o efeito de acumulação.

In [12]:
# Sinais isolados
s1, p1 = calcular_score(["evento_visao(stringing)", "criticidade(media)"], 0.65)
s2, p2 = calcular_score(["vibracao(warning)", "criticidade(media)"], 0.65)
s3, p3 = calcular_score(["prazo(curto)", "criticidade(media)"], 0.65)

# Todos combinados — similar ao F5 com criticidade média
fatos_combinados = [
    "evento_visao(stringing)",
    "vibracao(warning)",
    "criticidade(media)",
    "prazo(curto)",
]
s_total, p_total = calcular_score(fatos_combinados, 0.65)

print("EFEITO DE ACUMULAÇÃO DE SINAIS FRACOS")
print("=" * 50)
print(f"  Sinal 1 isolado (stringing):   score = {s1:3d}  →  {p1}")
print(f"  Sinal 2 isolado (vib warning): score = {s2:3d}  →  {p2}")
print(f"  Sinal 3 isolado (prazo curto): score = {s3:3d}  →  {p3}")
print(f"  TODOS COMBINADOS:              score = {s_total:3d}  →  {p_total}")

print("\nRodando o motor com todos combinados:")
preparar_e_rodar_fatos("Acumulação de sinais fracos", fatos_combinados, confianca_num=0.65, verbose=False)

print("\n📌 Nenhum sinal sozinho justificaria AJUSTAR.")
print("   Juntos, eles compõem um score que muda a decisão.")

EFEITO DE ACUMULAÇÃO DE SINAIS FRACOS
  Sinal 1 isolado (stringing):   score =  28  →  score_risco(baixo)
  Sinal 2 isolado (vib warning): score =  28  →  score_risco(baixo)
  Sinal 3 isolado (prazo curto): score =  18  →  score_risco(baixo)
  TODOS COMBINADOS:              score =  58  →  score_risco(alto)

Rodando o motor com todos combinados:

CASO: Acumulação de sinais fracos
Score calculado: 58  →  score_risco(alto)
Fatos na WM: ['evento_visao(stringing)', 'vibracao(warning)', 'criticidade(media)', 'prazo(curto)', 'score_risco(alto)']

--- RESUMO ---
🎯 Resultado: acao(AJUSTAR)

📌 Nenhum sinal sozinho justificaria AJUSTAR.
   Juntos, eles compõem um score que muda a decisão.


## 13. Exercício 1 — Calibrar os pesos do score

**Cenário:** a equipe de engenharia recebeu novos dados históricos de falhas e identificou que:
- `under_extrusion` estava sendo subestimado — deve ter peso **30** (e não 15)
- `prazo(curto)` estava sendo superestimado — deve ter peso **5** (e não 10)
- Um novo sinal `temperatura_alta` deve ser adicionado com peso **25**

**O que fazer:**
1. Atualize o dicionário `PESOS_SCORE_V2` com os novos valores
2. Adicione o predicado `temperatura_alta` ao dicionário
3. Rode o caso H1 abaixo e verifique se a decisão muda

⚠ **NÃO altere o motor nem a BK_GRADUAL.**

### 💡 DICA
- Copie `PESOS_SCORE` e ajuste apenas os valores indicados
- A lógica de faixas é a mesma de `calcular_score`
- Compare os scores antes/depois para entender o impacto

✅ Complete o código abaixo.

In [13]:
# ===== IMPLEMENTAR AQUI =====

# TODO 1: copie PESOS_SCORE e ajuste os valores indicados no enunciado
PESOS_SCORE_V2 = PESOS_SCORE.copy()
# PESOS_SCORE_V2["evento_visao(under_extrusion)"] = ???   # era 15, deve ser 30
# PESOS_SCORE_V2["prazo(curto)"] = ???                    # era 10, deve ser 5
# PESOS_SCORE_V2["temperatura_alta"] = ???                # novo sinal, peso 25

# TODO 2: crie calcular_score_v2 usando PESOS_SCORE_V2
# def calcular_score_v2(fatos: List[str], confianca_num: float = 0.0) -> Tuple[int, str]:
#     score = sum(PESOS_SCORE_V2.get(f, 0) for f in fatos)
#     if confianca_num >= 0.80:
#         score += PESOS_SCORE_V2["confianca_score_acima_0.80"]
#     score = min(score, 100)
#     if score < 30:   faixa = "baixo"
#     elif score < 50: faixa = "moderado"
#     elif score < 70: faixa = "alto"
#     elif score < 85: faixa = "critico"
#     else:            faixa = "extremo"
#     return score, f"score_risco({faixa})"

# Caso H1 — under_extrusion + temperatura alta + prazo curto
fatos_H1 = [
    "evento_visao(under_extrusion)",
    "confianca(baixa)",
    "temperatura_alta",
    "vibracao(normal)",
    "criticidade(alta)",
    "prazo(curto)",
]

print("Score com pesos originais:", calcular_score(fatos_H1, 0.78))
# print("Score com pesos v2:     ", calcular_score_v2(fatos_H1, 0.78))  # descomentar

Score com pesos originais: (45, 'score_risco(moderado)')


## 14. Exercício 2 — Criar uma nova regra gradual

**Cenário:** após análise operacional, a equipe identificou uma lacuna: quando há `under_extrusion` suspeito em peça de `criticidade(media)` com `score_risco(moderado)`, o sistema não produz nenhuma ação — fica sem decisão.

**Política nova definida pela engenharia:**
> Under-extrusion suspeito com risco moderado em peça de criticidade média → **AJUSTAR** temperatura de extrusão.

**O que fazer:**
1. Crie a regra `R18_AJUSTAR_UNDER_EXTRUSION_MODERADO` na célula abaixo
2. Adicione à BK estendida
3. Teste com o caso H2 fornecido
4. Use `por_que()` para verificar a cadeia causal

⚠ **NÃO altere o motor nem a BK_GRADUAL original.**

### 💡 DICA
- A regra de interpretação (`defeito_suspeito(under_extrusion)`) já existe na BK_GRADUAL como **R05**
- Você só precisa criar a regra de decisão
- Prioridade deve estar **entre MONITORAR (45) e INVESTIGAR (72)**

✅ Complete os TODOs abaixo.

In [14]:
# ===== IMPLEMENTAR AQUI =====

BK_EXT = BK_GRADUAL.copy()

# A regra de interpretação já existe na BK_GRADUAL:
# R05_UNDER_EXTRUSION_SUSPEITO: evento_visao(under_extrusion) + confianca(baixa) → defeito_suspeito(under_extrusion)
# Você só precisa criar a regra de decisão abaixo.

# TODO: regra de decisão (camada 2)
regra_ajustar_under_extrusion = Rule(
    nome="R18_AJUSTAR_UNDER_EXTRUSION_MODERADO",
    condicoes=[
        # TODO: completar condições
        # Dica: defeito_suspeito(under_extrusion) + score_risco(moderado) + criticidade(media)
    ],
    acao="",   # TODO: qual ação?
    prioridade=0,  # TODO: entre MONITORAR (45) e INVESTIGAR (72)
    justificativa="Under-extrusion suspeito com risco moderado em peça média → ajustar temperatura de extrusão."
)

BK_EXT.append(regra_ajustar_under_extrusion)

# Caso H2 — under_extrusion moderado em peça média
fatos_H2 = [
    "evento_visao(under_extrusion)",
    "confianca(baixa)",
    "vibracao(normal)",
    "criticidade(media)",
    "prazo(longo)",
]

eng_H2, res_H2 = preparar_e_rodar_fatos(
    "H2 — under_extrusion moderado",
    fatos_H2,
    confianca_num=0.70,
    regras=BK_EXT,
    verbose=True
)

print("\nÁrvore por_que('acao(AJUSTAR)'):")
# print(json.dumps(eng_H2.por_que("acao(AJUSTAR)"), indent=2, ensure_ascii=False))  # descomentar após implementar


CASO: H2 — under_extrusion moderado
Score calculado: 23  →  score_risco(baixo)
Fatos na WM: ['evento_visao(under_extrusion)', 'confianca(baixa)', 'vibracao(normal)', 'criticidade(media)', 'prazo(longo)', 'score_risco(baixo)']

Ciclo 1
Agenda: ['R05_UNDER_EXTRUSION_SUSPEITO', 'R20_MONITORAR_SCORE_BAIXO', 'R18_AJUSTAR_UNDER_EXTRUSION_MODERADO'] 
SELECT → R05_UNDER_EXTRUSION_SUSPEITO (P=55)
FIRE   → defeito_suspeito(under_extrusion)

Ciclo 2
Agenda: ['R20_MONITORAR_SCORE_BAIXO', 'R18_AJUSTAR_UNDER_EXTRUSION_MODERADO'] 
SELECT → R20_MONITORAR_SCORE_BAIXO (P=30)
FIRE   → acao(MONITORAR)

Ciclo 3
Agenda: ['R18_AJUSTAR_UNDER_EXTRUSION_MODERADO'] 
SELECT → R18_AJUSTAR_UNDER_EXTRUSION_MODERADO (P=0)
FIRE   → 

Ciclo 4: agenda vazia → convergiu.

--- RESUMO ---
🎯 Resultado: acao(MONITORAR)

Árvore por_que('acao(AJUSTAR)'):


## 15. Exercício 3 — Desafio: o score resolve o problema do limiar?

**Cenário reflexivo:** a Seção 5 mostrou que o limiar rígido criava decisões radicalmente diferentes para casos quase idênticos. O score suavizou isso — mas será que o eliminou?

**O que fazer:**
1. Crie dois casos baseados em F5 que diferem apenas em `confianca_num`: 0.795 e 0.805
2. Verifique se os scores são diferentes
3. Verifique se as decisões são diferentes
4. Responda: **o score elimina o problema do limiar ou apenas o desloca?**

✅ Implemente e responda na célula de texto abaixo.

In [15]:
# ===== INVESTIGAR AQUI =====

fatos_base = FATOS["F5"]  # stringing + warning + criticidade alta + prazo longo

s_795, p_795 = calcular_score(fatos_base, confianca_num=0.795)
s_805, p_805 = calcular_score(fatos_base, confianca_num=0.805)

print(f"confianca=0.795: score={s_795}  predicado={p_795}")
print(f"confianca=0.805: score={s_805}  predicado={p_805}")

# TODO: rode os dois casos e compare as decisões
# eng_795, res_795 = preparar_e_rodar("F5", confianca_num=0.795, verbose=False)
# eng_805, res_805 = preparar_e_rodar("F5", confianca_num=0.805, verbose=False)
# Responda: as decisões são iguais ou diferentes? O que isso significa?

confianca=0.795: score=60  predicado=score_risco(alto)
confianca=0.805: score=75  predicado=score_risco(critico)


### ✍ Resposta — Exercício 3

*(Escreva sua análise aqui)*

O score **elimina** o problema do limiar rígido? Ou apenas **desloca** o problema para o limiar entre faixas?

Considere:
- O que acontece exatamente na fronteira entre `score_risco(moderado)` e `score_risco(alto)`?
- Existe uma solução que **não** dependa de limiar algum?
- Qual seria o próximo passo natural para resolver isso? *(dica: próxima aula)*

## 16. Fechamento

Hoje vimos três ideias centrais:

1. **Decisão gradual**: entre GO e NOGO há um espectro — MONITORAR e AJUSTAR são respostas legítimas e frequentemente mais corretas que parar.

2. **Score como mediador**: sinais fracos se acumulam. Um score ponderado captura risco composto que nenhum sensor isolado expressaria.

3. **Contexto sempre importa**: o mesmo defeito exige respostas diferentes dependendo de criticidade e prazo.

---

### O limite que encontramos hoje

O Exercício 3 revelou que o score apenas **desloca** o problema do limiar — agora ele está na fronteira entre faixas.

Isso nos leva à questão da próxima aula:

> **E se o próprio limiar também fosse gradual?**
> Como modelar 'quase alto', 'levemente crítico' e 'bem provável'?

---

## Pergunta final (discursiva)

Responda em 5–8 linhas:

1. Por que adicionar MONITORAR e AJUSTAR torna o sistema **mais seguro** — e não apenas mais complexo?
2. Em que situação você **não** usaria um score acumulado?
3. Como você justificaria os pesos do score para um auditor externo?

> Dica: use as palavras **proporcionalidade**, **rastreabilidade**, **domínio** e **limiar**.